# Enterprise Generative AI

Generative AI is a type of artificial intelligence that can create new content and ideas, including conversations, stories, images, videos, and music. Like all artificial intelligence, generative AI is powered by machine learning models—very large models that are pre-trained on vast amounts of data and commonly referred to as Foundation Models (FMs). Apart from content creation, generative AI is also used to improve the quality of digital images, edit video, build prototypes quickly for manufacturing, augment data with synthetic datasets, and more.

## Using with Falcon model 

---
In this prod notebook, we demonstrate how to use the SageMaker Python SDK to deploy Falcon models for text generation. It is a permissively licensed ([Apache-2.0](https://jumpstart-cache-prod-us-east-2.s3.us-east-2.amazonaws.com/licenses/Apache-License/LICENSE-2.0.txt)) open source model trained on the [RefinedWeb dataset](https://huggingface.co/datasets/tiiuae/falcon-refinedweb). We show several example use cases including code generation, question answering, translation etc.

---

### About the model

---
Falcon is a causal decoder-only model built by [Technology Innovation Institute](https://www.tii.ae/) (TII) and trained on more than 1 trillion tokens of RefinedWeb enhanced with curated corpora. It was built using custom-built tooling for data pre-processing and model training built on Amazon SageMaker. It features an architecture optimized for inference, with FlashAttention and multiquery. 


[Refined Web Dataset](https://huggingface.co/datasets/tiiuae/falcon-refinedweb): Falcon RefinedWeb is a massive English web dataset built by TII and released under an Apache 2.0 license. It is a highly filtered dataset with large scale de-duplication of CommonCrawl. It is observed that models trained on RefinedWeb achieve performance equal to or better than performance achieved by training model on curated datasets, while only relying on web data.

**Model Sizes:**
- **Falcon-7b**: It is a 7 billion parameter model trained on 1.5 trillion tokens.
- **Falcon-40B**: It is a 40 billion parameter model trained on 1 trillion tokens.  It has surpassed renowned models like LLaMA-65B, StableLM, RedPajama and MPT on the public leaderboard maintained by Hugging Face, demonstrating its exceptional performance without specialized fine-tuning. To see comparison, see [OpenLLM Leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard). 

**Instruct models (Falcon-7b-instruct/Falcon-40B-instruct):** Instruct models are base falcon models fine-tuned on a mixture of chat and instruction datasets. They are ready-to-use chat/instruct models.  To use these models, please select `model_id` in the cell above to be "huggingface-textgeneration-falcon-7b-instruct-bf16" or "huggingface-textgeneration-falcon-40b-instruct-bf16".

It is [recommended](https://huggingface.co/tiiuae/falcon-7b) that Instruct models should be used without fine-tuning and base models should be fine-tuned further on the specific task.

**Limitations:**

- Falcon models are mostly trained on English data and may not generalize to other languages. 
- Falcon carries the stereotypes and biases commonly encountered online and in the training data. Hence, it is recommended to develop guardrails and to take appropriate precautions for any production use. This is a raw, pretrained model, which should be further finetuned for most usecases.


# Create a New JupyterLab with the Correct Image

However, this notebook was developed for:

- **SageMaker Distribution 3.4.2** ✅

Since different SageMaker Distribution versions contain different Python packages and SDK versions, the required module (`sagemaker.jumpstart`) is not available in version **4.3.1**.

---

## Steps to Fix the Issue

### Step 1
Open **Amazon SageMaker AI Studio**.

### Step 2
Click **JupyterLab**.

### Step 3
Create a **new JupyterLab Space** (or create a new JupyterLab application).

### Step 4
When prompted to choose an image:

- Select **SageMaker Distribution**
- Choose **Version 3.4.2**

### Step 5
Wait for the JupyterLab environment to start.

### Step 6
Upload or open the notebook.

### Step 7
Run the notebook from the beginning (**Run All Cells**).

---

## Expected Result

The following import should now execute successfully:

```python
from sagemaker.jumpstart.model import JumpStartModel
```

and the deployment code should run without the previous **ModuleNotFoundError**.

---

## Root Cause

The notebook was written for **SageMaker Distribution 3.4.2**, but it was executed in **SageMaker Distribution 4.3.1**.

Because the installed SageMaker SDK differs between these environments, the required `sagemaker.jumpstart` module was unavailable, resulting in the error.

## Step 0:- SageMaker Studio Environment and Its Resources

In [9]:
# ============================================================
# SAGEMAKER STUDIO ENVIRONMENT INFORMATION
# ============================================================

import boto3

# ------------------------------------------------------------
# 1. AWS REGION
# ------------------------------------------------------------

aws_region = boto3.Session().region_name

print("=" * 70)
print("SAGEMAKER STUDIO ENVIRONMENT")
print("=" * 70)

print("\nAWS Region:", aws_region)


# ------------------------------------------------------------
# 2. SAGEMAKER CLIENT
# ------------------------------------------------------------

sagemaker_client = boto3.client(
    "sagemaker",
    region_name=aws_region
)


# ------------------------------------------------------------
# 3. SAGEMAKER DOMAINS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. SAGEMAKER STUDIO DOMAINS")
print("=" * 70)

domains_response = sagemaker_client.list_domains()

domains = domains_response.get("Domains", [])

if domains:

    for domain in domains:

        print("\nDomain Name :", domain.get("DomainName"))
        print("Domain ID   :", domain.get("DomainId"))
        print("Status      :", domain.get("Status"))
        print("Created     :", domain.get("CreationTime"))

else:

    print("No SageMaker Studio Domain found.")


# ------------------------------------------------------------
# 4. USER PROFILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. USER PROFILES")
print("=" * 70)

user_profiles_response = sagemaker_client.list_user_profiles()

user_profiles = user_profiles_response.get(
    "UserProfiles",
    []
)

if user_profiles:

    for user in user_profiles:

        print("\nUser Profile Name :",
              user.get("UserProfileName"))

        print("User Profile ARN  :",
              user.get("UserProfileArn"))

        print("Status            :",
              user.get("Status"))

else:

    print("No User Profiles found.")


# ------------------------------------------------------------
# 5. SPACES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. SAGEMAKER SPACES")
print("=" * 70)

spaces_response = sagemaker_client.list_spaces()

spaces = spaces_response.get(
    "Spaces",
    []
)

if spaces:

    for space in spaces:

        print("\nSpace Name :",
              space.get("SpaceName"))

        print("Status     :",
              space.get("Status"))

        print("Domain ID  :",
              space.get("DomainId"))

else:

    print("No SageMaker Spaces found.")


# ------------------------------------------------------------
# 6. APPLICATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. SAGEMAKER APPLICATIONS")
print("=" * 70)

apps_response = sagemaker_client.list_apps()

apps = apps_response.get(
    "Apps",
    []
)

if apps:

    for app in apps:

        print("\nApp Name      :",
              app.get("AppName"))

        print("App Type      :",
              app.get("AppType"))

        print("Status        :",
              app.get("Status"))

        print("Domain ID     :",
              app.get("DomainId"))

        print("User Profile  :",
              app.get("UserProfileName"))

        print("Space Name    :",
              app.get("SpaceName"))

        resource_spec = app.get(
            "ResourceSpec",
            {}
        )

        print("Instance Type :",
              resource_spec.get("InstanceType"))

else:

    print("No SageMaker Applications found.")


# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. SUMMARY")
print("=" * 70)

print("\nAWS Region              :", aws_region)
print("Number of Domains       :", len(domains))
print("Number of User Profiles :", len(user_profiles))
print("Number of Spaces        :", len(spaces))
print("Number of Applications  :", len(apps))

print("\n" + "=" * 70)
print("✅ SageMaker environment information retrieved.")
print("=" * 70)

SAGEMAKER STUDIO ENVIRONMENT

AWS Region: us-east-1

1. SAGEMAKER STUDIO DOMAINS

Domain Name : QuickSetupDomain-20260817T183770
Domain ID   : d-soq4fkuoq8fr
Status      : InService
Created     : 2026-08-17 13:07:34.098000+00:00

2. USER PROFILES

User Profile Name : default-20260817T183770
User Profile ARN  : None
Status            : InService

3. SAGEMAKER SPACES

Space Name : GenAi
Status     : InService
Domain ID  : d-soq4fkuoq8fr

4. SAGEMAKER APPLICATIONS

App Name      : default
App Type      : JupyterLab
Status        : InService
Domain ID     : d-soq4fkuoq8fr
User Profile  : None
Space Name    : GenAi
Instance Type : ml.t3.large

5. SUMMARY

AWS Region              : us-east-1
Number of Domains       : 1
Number of User Profiles : 1
Number of Spaces        : 1
Number of Applications  : 1

✅ SageMaker environment information retrieved.


## Step 2 — Get the SageMaker execution role

In [10]:
# ============================================================
# STEP 2 — GET SAGEMAKER EXECUTION ROLE
# ============================================================

import boto3

aws_region = boto3.Session().region_name

sagemaker_client = boto3.client(
    "sagemaker",
    region_name=aws_region
)

domain_id = "d-soq4fkuoq8fr"

# Get Domain information
domain_response = sagemaker_client.describe_domain(
    DomainId=domain_id
)

print("=" * 70)
print("SAGEMAKER DOMAIN INFORMATION")
print("=" * 70)

print("\nDomain Name:")
print(domain_response.get("DomainName"))

print("\nDomain ID:")
print(domain_response.get("DomainId"))

print("\nStatus:")
print(domain_response.get("Status"))

print("\nDefault Execution Role:")
execution_role = domain_response.get("DefaultUserSettings", {}).get(
    "ExecutionRole"
)

print(execution_role)

print("\n" + "=" * 70)

SAGEMAKER DOMAIN INFORMATION

Domain Name:
QuickSetupDomain-20260817T183770

Domain ID:
d-soq4fkuoq8fr

Status:
InService

Default Execution Role:
arn:aws:iam::472360887517:role/service-role/AmazonSageMaker-ExecutionRole-20260817T183771



## STEP 3 — CHECK SAGEMAKER GPU QUOTAS

In [14]:
# ============================================================
# STEP 3 — CHECK SAGEMAKER GPU QUOTAS
# ============================================================

import boto3

quota_client = boto3.client(
    "service-quotas",
    region_name="us-east-1"
)

quotas = quota_client.list_service_quotas(
    ServiceCode="sagemaker"
)["Quotas"]

print("=" * 70)
print("SAGEMAKER GPU / INSTANCE QUOTAS")
print("=" * 70)

for quota in quotas:

    name = quota.get("QuotaName", "")

    if any(x in name.lower() for x in [
        "g5",
        "gpu",
        "ml.g"
    ]):

        print("\nQuota:")
        print(name)

        print("Value:")
        print(quota.get("Value"))

        print("Quota Code:")
        print(quota.get("QuotaCode"))

        print("-" * 70)

SAGEMAKER GPU / INSTANCE QUOTAS

Quota:
ml.g6.16xlarge for endpoint usage
Value:
0.0
Quota Code:
L-913947FA
----------------------------------------------------------------------


In [13]:
import boto3

quota_client = boto3.client(
    "service-quotas",
    region_name="us-east-1"
)

print("=" * 70)
print("SAGEMAKER ENDPOINT QUOTA CHECK")
print("=" * 70)

response = quota_client.list_service_quotas(
    ServiceCode="sagemaker"
)

for quota in response["Quotas"]:
    print(quota["QuotaName"])

SAGEMAKER ENDPOINT QUOTA CHECK
Studio CodeEditor Apps running on ml.r6id.large instances
ml.c4.xlarge for spot training job usage
Studio CodeEditor Apps running on ml.r5.4xlarge instances
ml.r5d.xlarge for training job usage
ml.g6.16xlarge for endpoint usage
ml.m6i.8xlarge for cluster spot instance usage
ml.i3en.16xlarge for cluster spot instance usage
ml.c6i.2xlarge for cluster spot instance usage
ml.r5d.24xlarge for spot training job usage
Total number of trials a single trial component can be associated to
ml.r7i.16xlarge for cluster usage
ml.c7i.12xlarge for endpoint usage
Studio KernelGateway Apps running on ml.m5d.12xlarge instance
ml.c4.xlarge for training job usage
Studio JupyterLab Apps running on ml.m7i.2xlarge instances
ml.c4.8xlarge for notebook instance usage
ml.trn1n.32xlarge for training job usage
ml.m5d.xlarge for notebook instance usage
ml.m6i.24xlarge for cluster spot instance usage


In [15]:
import boto3

quota_client = boto3.client(
    "service-quotas",
    region_name="us-east-1"
)

quotas = quota_client.list_service_quotas(
    ServiceCode="sagemaker"
)["Quotas"]

print("=" * 70)
print("SAGEMAKER ENDPOINT QUOTAS")
print("=" * 70)

found = False

for quota in quotas:

    name = quota["QuotaName"]

    if "endpoint usage" in name.lower():

        found = True

        print("\nQuota Name :", name)
        print("Value      :", quota["Value"])
        print("Quota Code :", quota["QuotaCode"])

        print("-" * 70)

if not found:
    print("\nNo endpoint quotas were returned.")

SAGEMAKER ENDPOINT QUOTAS

Quota Name : ml.g6.16xlarge for endpoint usage
Value      : 0.0
Quota Code : L-913947FA
----------------------------------------------------------------------

Quota Name : ml.c7i.12xlarge for endpoint usage
Value      : 0.0
Quota Code : L-D98C40FA
----------------------------------------------------------------------


## Step 1 : Upgrade SageMaker 

In [2]:
# Skip this upgrade.
# !pip install sagemaker --quiet --upgrade --force-reinstall

## Step 2 : Define the Model
Define the **huggingface-llm-falcon-7b-instruct-bf16** model

In [16]:
from sagemaker.jumpstart.model import JumpStartModel

In [17]:
model_id, model_version = (
    "huggingface-llm-falcon-7b-instruct-bf16",
    "*",
)

print(model_id)
print(model_version)

huggingface-llm-falcon-7b-instruct-bf16
*


## Step 3 : Deploy the model in Amazon SageMaker Inference Endpoint.

- Deploy Falcon 7B on `ml.g5.xlarge` GPU instance
- `ml.g5.xlarge` provides 1 NVIDIA A10G GPU and is cheaper than  `ml.g5.2xlarge`

In [18]:
from sagemaker.jumpstart.model import JumpStartModel

model_id = "huggingface-llm-falcon-7b-instruct-bf16"

my_model = JumpStartModel(
    model_id=model_id,
    instance_type="ml.g5.xlarge"
)

predictor = my_model.deploy(
    initial_instance_count=1
)

print("====================================")
print("✅ MODEL DEPLOYED")
print("====================================")
print("Predictor:", predictor)

Using model 'huggingface-llm-falcon-7b-instruct-bf16' with wildcard version identifier '*'. You can pin to version '4.6.16' for more stable results. Note that models may have different input/output signatures after a major version upgrade.
INFO:sagemaker:Creating model with name: hf-llm-falcon-7b-instruct-bf16-2026-08-18-11-37-14-665
INFO:sagemaker:Creating endpoint-config with name hf-llm-falcon-7b-instruct-bf16-2026-08-18-11-37-14-667
INFO:sagemaker:Creating endpoint with name hf-llm-falcon-7b-instruct-bf16-2026-08-18-11-37-14-667


-------------------!====================================
✅ MODEL DEPLOYED
Predictor: Predictor: {'endpoint_name': 'hf-llm-falcon-7b-instruct-bf16-2026-08-18-11-37-14-667', 'sagemaker_session': <sagemaker.session.Session object at 0x7f516ac14950>, 'serializer': <sagemaker.base_serializers.JSONSerializer object at 0x7f516b8d95e0>, 'deserializer': <sagemaker.base_deserializers.JSONDeserializer object at 0x7f516a4849b0>, '_content_type': 'application/json', '_accept': 'application/json'}


Model

   ↓
   
SageMaker Endpoint Configuration

   ↓
   
SageMaker Real-Time Endpoint


Yes! 🎉 This is progress. Your deployment has started successfully.

- That means SageMaker has accepted your deployment request and is now trying to provision the ml.g5.xlarge endpoint.

## Step 4 : Test the deployed model

The model supports a variety of actions like code generation, sentiment analysis, question answering, and summarization.

### Supported parameters

***
Some of the supported parameters while performing inference are the following:

* **max_length:** Model generates text until the output length (which includes the input context length) reaches `max_length`. If specified, it must be a positive integer.
* **max_new_tokens:** Model generates text until the output length (excluding the input context length) reaches `max_new_tokens`. If specified, it must be a positive integer.
* **num_beams:** Number of beams used in the greedy search. If specified, it must be integer greater than or equal to `num_return_sequences`.
* **no_repeat_ngram_size:** Model ensures that a sequence of words of `no_repeat_ngram_size` is not repeated in the output sequence. If specified, it must be a positive integer greater than 1.
* **temperature:** Controls the randomness in the output. Higher temperature results in output sequence with low-probability words and lower temperature results in output sequence with high-probability words. If `temperature` -> 0, it results in greedy decoding. If specified, it must be a positive float.
* **early_stopping:** If True, text generation is finished when all beam hypotheses reach the end of sentence token. If specified, it must be boolean.
* **do_sample:** If True, sample the next word as per the likelihood. If specified, it must be boolean.
* **top_k:** In each step of text generation, sample from only the `top_k` most likely words. If specified, it must be a positive integer.
* **top_p:** In each step of text generation, sample from the smallest possible set of words with cumulative probability `top_p`. If specified, it must be a float between 0 and 1.
* **return_full_text:** If True, input text will be part of the output generated text. If specified, it must be boolean. The default value for it is False.
* **stop**: If specified, it must a list of strings. Text generation stops if any one of the specified strings is generated.

We may specify any subset of the parameters mentioned above while invoking an endpoint. 

For more parameters and information on HF LLM DLC, please see [this article](https://huggingface.co/blog/sagemaker-huggingface-llm#4-run-inference-and-chat-with-our-model).
***

### Step 4.1: Test endpoint

In [28]:
%%time

import time
from datetime import datetime

# ============================================================
# STEP 1 — START
# ============================================================

start_time = time.time()

print("=" * 70)
print("🚀 STARTING SAGEMAKER INFERENCE")
print("=" * 70)

print(f"⏰ Time       : {datetime.now().strftime('%H:%M:%S')}")
print("📌 Endpoint   :", predictor.endpoint_name)
print("🧠 Model      : Falcon 7B Instruct")
print("💻 Instance   : ml.g5.xlarge")

# ============================================================
# STEP 2 — PREPARE PROMPT
# ============================================================

prompt = "Tell me about Amazon SageMaker."

print("\n" + "-" * 70)
print("📝 STEP 1: Preparing prompt")
print("-" * 70)

print("Prompt:")
print(prompt)

# ============================================================
# STEP 3 — CREATE PAYLOAD
# ============================================================

payload = {
    "inputs": prompt,
    "parameters": {
        "do_sample": True,
        "top_p": 0.9,
        "temperature": 0.1,
        "max_new_tokens": 1024,
        "stop": ["<|endoftext|>", "</s>"]
    }
}

print("\n" + "-" * 70)
print("📦 STEP 2: Payload prepared")
print("-" * 70)

print("Generation parameters:")
print("  Temperature     :", payload["parameters"]["temperature"])
print("  Top P           :", payload["parameters"]["top_p"])
print("  Max New Tokens  :", payload["parameters"]["max_new_tokens"])

# ============================================================
# STEP 4 — SEND REQUEST
# ============================================================

print("\n" + "-" * 70)
print("📡 STEP 3: Sending request to SageMaker endpoint...")
print("-" * 70)

print("⏳ Waiting for Falcon 7B response...")
print("⚠️  First inference may take some time.")

request_start = time.time()

try:

    response = predictor.predict(payload)

    request_end = time.time()

    print("\n✅ Request completed!")

    print(
        f"⏱️ Inference time: "
        f"{request_end - request_start:.2f} seconds"
    )

except Exception as e:

    print("\n❌ INFERENCE FAILED")
    print("Error:", str(e))

    raise

# ============================================================
# STEP 5 — DISPLAY RESPONSE
# ============================================================

print("\n" + "=" * 70)
print("🤖 MODEL RESPONSE")
print("=" * 70)

try:

    generated_text = response[0]["generated_text"]

except (KeyError, IndexError, TypeError):

    generated_text = response

print(generated_text)

# ============================================================
# STEP 6 — TOTAL TIME
# ============================================================

total_time = time.time() - start_time

print("\n" + "=" * 70)
print("✅ INFERENCE COMPLETED")
print("=" * 70)

print(f"⏱️ Total execution time: {total_time:.2f} seconds")
print(f"⏰ Finished at: {datetime.now().strftime('%H:%M:%S')}")

🚀 STARTING SAGEMAKER INFERENCE
⏰ Time       : 11:47:29
📌 Endpoint   : hf-llm-falcon-7b-instruct-bf16-2026-08-18-11-37-14-667
🧠 Model      : Falcon 7B Instruct
💻 Instance   : ml.g5.xlarge

----------------------------------------------------------------------
📝 STEP 1: Preparing prompt
----------------------------------------------------------------------
Prompt:
Tell me about Amazon SageMaker.

----------------------------------------------------------------------
📦 STEP 2: Payload prepared
----------------------------------------------------------------------
Generation parameters:
  Temperature     : 0.1
  Top P           : 0.9
  Max New Tokens  : 1024

----------------------------------------------------------------------
📡 STEP 3: Sending request to SageMaker endpoint...
----------------------------------------------------------------------
⏳ Waiting for Falcon 7B response...
⚠️  First inference may take some time.

✅ Request completed!
⏱️ Inference time: 2.18 seconds

🤖 MODEL RESP

Excellent! 🎉 Your Falcon 7B SageMaker endpoint is working successfully.

### Step 4.2 : Test use cases

Let's define a function to query endpoint.

In [29]:
def query_endpoint(payload):
    """Query the SageMaker Falcon 7B endpoint and print the response."""

    print("\n" + "=" * 70)
    print("📡 Sending request to SageMaker...")
    print("=" * 70)

    response = predictor.predict(payload)

    print("\033[1mInput:\033[0m")
    print(payload["inputs"])

    print("\n\033[1mOutput:\033[0m")

    # Falcon/TGI response
    if isinstance(response, list):
        print(response[0]["generated_text"])
    else:
        print(response)

    print("=" * 70)

### Step 4.2.1 : Test code generation

In [30]:
payload = {
    "inputs": "Write a program to compute factorial in python from number 4. Give result:",
    "parameters": {
        "max_new_tokens": 200
    }
}

query_endpoint(payload)


📡 Sending request to SageMaker...
Input:
Write a program to compute factorial in python from number 4. Give result:

Output:
Write a program to compute factorial in python from number 4. Give result:
Here's a Python program to compute the factorial of a number from 4:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)

print(factorial(4)) # Output: 4 * 3 * 2 * 1 = 24


### Step 4.2.2 : Test sentence completion

In [31]:
payload = {
    "inputs": "Building a website can be done in 7 simple steps:",
    "parameters": {
        "max_new_tokens": 110,
        "no_repeat_ngram_size": 3
    }
}

query_endpoint(payload)


📡 Sending request to SageMaker...
Input:
Building a website can be done in 7 simple steps:

Output:
Building a website can be done in 7 simple steps:
1. Choose a domain name
2. Select a web hosting provider
3. Design your website
4. Develop your website
5. Test your website
6. Optimize your website
7. Maintain your website
Choosing a domain name is the first step in building a website. You can choose a domain name that is related to your website or business. You can also choose a domain name that is easy to remember and easy to spell.
Once you have chosen a domain name, you will need to select a web hosting provider.


### Step 4.2.3 : Test translation

In [32]:
# Few-shot prompting: provide examples to guide the model
payload = {
    "inputs": """Translate English to French:

sea otter => loutre de mer

peppermint => menthe poivrée

plush girafe => girafe peluche

butter =>""",
    
    # Limit the generated response to 3 tokens
    "parameters": {
        "max_new_tokens": 3
    }
}

# Send the prompt to the SageMaker Falcon 7B endpoint
query_endpoint(payload)


📡 Sending request to SageMaker...
Input:
Translate English to French:

sea otter => loutre de mer

peppermint => menthe poivrée

plush girafe => girafe peluche

butter =>

Output:
Translate English to French:

sea otter => loutre de mer

peppermint => menthe poivrée

plush girafe => girafe peluche

butter => beurre



### Step 4.2.4 : Test sentiment analysis

In [9]:
payload = {
    "inputs": """"I hate it when my phone battery dies."
                Sentiment: Negative
                ###
                Tweet: "My day has been :+1:"
                Sentiment: Positive
                ###
                Tweet: "This is the link to the article"
                Sentiment: Neutral
                ###
                Tweet: "This new halal music video was incredibile"
                Sentiment:""",
    "parameters": {
        "max_new_tokens":2
    }
}
query_endpoint(payload)

 Input: "I hate it when my phone battery dies."
                Sentiment: Negative
                ###
                Tweet: "My day has been :+1:"
                Sentiment: Positive
                ###
                Tweet: "This is the link to the article"
                Sentiment: Neutral
                ###
                Tweet: "This new halal music video was incredibile"
                Sentiment:
 Output: "I hate it when my phone battery dies."
                Sentiment: Negative
                ###
                Tweet: "My day has been :+1:"
                Sentiment: Positive
                ###
                Tweet: "This is the link to the article"
                Sentiment: Neutral
                ###
                Tweet: "This new halal music video was incredibile"
                Sentiment: Positive
                


### Step 4.2.5 : Test question answering

In [33]:
# Simple question-answering prompt
payload = {
    "inputs": "Could you remind me when was the Python programming language invented?",
    
    # Limit the generated response to 50 tokens
    "parameters": {
        "max_new_tokens": 50
    }
}

# Send the prompt to the SageMaker endpoint
query_endpoint(payload)


📡 Sending request to SageMaker...
Input:
Could you remind me when was the Python programming language invented?

Output:
Could you remind me when was the Python programming language invented?
The Python programming language was invented in 1985 by Guido Van Rossum.


### Step 4.2.6 : Test recipe generation

In [34]:
# Recipe generation prompt
payload = {
    "inputs": "What is the recipe for a delicious lemon cheesecake?",
    
    # Allow up to 400 tokens for the recipe
    "parameters": {
        "max_new_tokens": 400
    }
}

# Send the prompt to the SageMaker endpoint
query_endpoint(payload)


📡 Sending request to SageMaker...
Input:
What is the recipe for a delicious lemon cheesecake?

Output:
What is the recipe for a delicious lemon cheesecake?
Here is a recipe for a delicious lemon cheesecake:

Ingredients:
- 1 1/2 cups graham cracker crumbs
- 4 tablespoons butter, melted
- 2 (8 ounce) packages cream cheese, softened
- 1/2 cup granulated sugar
- 2 eggs
- 1/2 cup lemon juice
- 1/2 teaspoon salt
- 1/2 teaspoon vanilla extract
- 1/2 cup heavy cream
- 1/2 cup granulated sugar
- 1/2 teaspoon lemon zest

Instructions:
1. Preheat oven to 350 degrees F.
2. In a medium bowl, mix together the graham cracker crumbs and melted butter. Press the mixture onto the bottom and sides of a 9-inch springform pan.
3. In a large bowl, beat the cream cheese and sugar until smooth. Add the eggs, lemon juice, salt, vanilla, and heavy cream. Beat until well combined.
4. Pour the mixture into the prepared pan.
5. Bake for 30 minutes or until the cheesecake is set.
6. Let cool for 10 minutes before

### Step 4.2.7 : Test summarization

In [35]:
payload = {
    "inputs":"""Amazon SageMaker is a fully managed machine learning service. With SageMaker, 
    data scientists and developers can quickly and easily build and train machine learning models, 
    and then directly deploy them into a production-ready hosted environment. It provides an 
    integrated Jupyter authoring notebook instance for easy access to your data sources for 
    exploration and analysis, so you don't have to manage servers. It also provides common 
    machine learning algorithms that are optimized to run efficiently against extremely 
    large data in a distributed environment. With native support for bring-your-own-algorithms 
    and frameworks, SageMaker offers flexible distributed training options that adjust to your 
    specific workflows. Deploy a model into a secure and scalable environment by launching it 
    with a few clicks from SageMaker Studio or the SageMaker console. Summarize the article above:""",
    "parameters":{
        "max_new_tokens":200
        }
    }
query_endpoint(payload)


📡 Sending request to SageMaker...
Input:
Amazon SageMaker is a fully managed machine learning service. With SageMaker, 
    data scientists and developers can quickly and easily build and train machine learning models, 
    and then directly deploy them into a production-ready hosted environment. It provides an 
    integrated Jupyter authoring notebook instance for easy access to your data sources for 
    exploration and analysis, so you don't have to manage servers. It also provides common 
    machine learning algorithms that are optimized to run efficiently against extremely 
    large data in a distributed environment. With native support for bring-your-own-algorithms 
    and frameworks, SageMaker offers flexible distributed training options that adjust to your 
    specific workflows. Deploy a model into a secure and scalable environment by launching it 
    with a few clicks from SageMaker Studio or the SageMaker console. Summarize the article above:

Output:
Amazon SageMaker

### 5. Clean up the endpoint - Do not run below codes.

In [ ]:
# Delete the SageMaker endpoint
# predictor.delete_model()
# predictor.delete_endpoint()